<a href="https://colab.research.google.com/github/chiyanglin-AStar/Python_3D_Lib/blob/main/Jax_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## JAX Tutorial in Google Colab

JAX is a high-performance numerical computing library, designed for high-performance machine learning research. It combines **automatic differentiation (Autograd)** for differentiating native Python and NumPy functions, with **XLA (Accelerated Linear Algebra)** for compiling and running your NumPy programs on GPUs and TPUs. This allows JAX to achieve impressive performance and flexibility, especially for deep learning.

### Key Features of JAX:

1.  **`jax.grad`**: Automatic differentiation for scalar-valued functions.
2.  **`jax.jit`**: Just-in-time compilation for accelerating functions.
3.  **`jax.vmap`**: Automatic vectorization (batching) of functions.
4.  **`jax.pmap`**: Automatic parallelization across multiple devices.

### Installation Steps

First, we need to install JAX and `jaxlib`. The installation command depends on whether you want CPU, GPU, or TPU support. Colab environments typically offer GPU or TPU runtimes, which can be selected from `Runtime -> Change runtime type`.

For GPU support (recommended in Colab if available):

In [ ]:
# Uninstall existing JAX/jaxlib to ensure a clean installation
!pip uninstall -y jax jaxlib

# Install the latest JAX from PyPI. This will install the CPU-only version initially.
!pip install -q jax

# Install jaxlib for CUDA 12 from Google's releases.
# We use `--upgrade` to ensure it can upgrade `jax` if `jaxlib` has a tighter dependency on a newer `jax` version.
# `--no-index` is crucial here to prevent pip from installing a CPU-only jaxlib from PyPI.
!pip install -q --upgrade jaxlib -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html --no-index

# Also install optax and tensorflow_datasets for later parts of the tutorial
!pip install -qqq optax
!pip install -qqq tensorflow
!pip install -qqq tensorflow_datasets

print("JAX, jaxlib, optax, tensorflow, and tensorflow_datasets installation initiated.")

### Verify Installation

After installation, it's good practice to verify that JAX is correctly installed and can detect your hardware accelerator (GPU or TPU if available).

### 1. `jax.grad`: 自動微分 (Automatic Differentiation)

`jax.grad` 是 JAX 最核心的功能之一，它允許我們輕鬆地計算 Python 函數的梯度。這在訓練神經網絡和其他優化問題中非常有用。

我們將定義一個簡單的函數，然後使用 `jax.grad` 來計算它的導數。

### 2. `jax.jit`: Just-In-Time 編譯

`jax.jit` 是一個函數轉換 (function transformation)，它會將您的 Python/NumPy 函數即時 (Just-In-Time) 編譯成 XLA (Accelerated Linear Algebra) 優化過的程式碼。這可以顯著提升函數的執行速度，尤其是在重複執行相同計算、且輸入形狀 (shape) 不變的情況下。當您在 GPU 或 TPU 上運行 JAX 時，`jax.jit` 的加速效果會更加明顯。

`jax.jit` 的主要優點：
-   **效能提升**：通過 XLA 編譯器，將多個 JAX 操作融合 (fuse) 成一個優化過的大型操作，減少了 Python 的解釋器開銷。
-   **設備加速**：充分利用 GPU 和 TPU 的並行計算能力。
-   **首次運行開銷**：第一次執行 `jitted` 函數時會有編譯開銷，之後的調用會非常快。

**使用注意事項**：
-   `jax.jit` 要求函數的輸入形狀通常保持不變。如果輸入形狀經常變化，每次變換都將重新編譯，可能導致性能下降。
-   函數內部應盡量避免 Python 的控制流（如 `if/else`, `for` 迴圈），除非這些控制流可以被 JAX 的控制流原語（如 `jax.lax.cond`, `jax.lax.while_loop`）靜態化或轉換。

我們將使用一個簡單的矩陣乘法範例來展示 `jax.jit` 的加速效果。

In [ ]:
import jax
import jax.numpy as jnp
import time

# Define a function without JIT
def matrix_multiply(x, y):
    return jnp.dot(x, y)

# Define the same function with JIT
@jax.jit
def jitted_matrix_multiply(x, y):
    return jnp.dot(x, y)

# Create large matrices
key = jax.random.PRNGKey(0)
size = 1000
matrix_a = jax.random.normal(key, (size, size))
matrix_b = jax.random.normal(key, (size, size))

print(f"Performing {size}x{size} matrix multiplication...")

# Measure execution time without JIT
start_time = time.time()
result_non_jit = matrix_multiply(matrix_a, matrix_b).block_until_ready() # .block_until_ready() ensures computation completes
end_time = time.time()
print(f"Time without JIT: {end_time - start_time:.4f} seconds")

# Measure execution time with JIT
# First run includes compilation time
start_time = time.time()
result_jit_first = jitted_matrix_multiply(matrix_a, matrix_b).block_until_ready()
end_time = time.time()
print(f"Time with JIT (first run, including compilation): {end_time - start_time:.4f} seconds")

# Subsequent runs are much faster as the function is already compiled
start_time = time.time()
result_jit_second = jitted_matrix_multiply(matrix_a, matrix_b).block_until_ready()
end_time = time.time()
print(f"Time with JIT (second run, compiled): {end_time - start_time:.4f} seconds")

# Verify results are the same
print(f"Results are approximately equal: {jnp.allclose(result_non_jit, result_jit_first)}")

In [ ]:
import jax
import jax.numpy as jnp

# Define a simple scalar function
def my_function(x):
    return x**2 + 2 * x + 1

# Calculate the gradient of my_function using jax.grad
# jax.grad returns a function that computes the gradient of the original function.
# The derivative of x**2 + 2x + 1 is 2x + 2.
grad_my_function = jax.grad(my_function)

# Test the original function
x_val = 3.0
print(f"Original function output for x={x_val}: {my_function(x_val)}")

# Test the gradient function
# For x=3.0, the gradient should be 2*3 + 2 = 8.0
print(f"Gradient of my_function at x={x_val}: {grad_my_function(x_val)}")


# Example with a vector input (gradient will be a vector - Jacobian-vector product if not reduced)
# Let's consider a function that sums squares of elements
def sum_of_squares(x):
    return jnp.sum(x**2)

# The gradient for sum_of_squares(x1, x2, ...) = x1^2 + x2^2 + ...
# with respect to each xi is 2*xi.

grad_sum_of_squares = jax.grad(sum_of_squares)

x_vec = jnp.array([1.0, 2.0, 3.0])
print(f"\nOriginal vector function output for x={x_vec}: {sum_of_squares(x_vec)}")
print(f"Gradient of sum_of_squares at x={x_vec}: {grad_sum_of_squares(x_vec)}")


In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Jaxlib version: {jax.lib.__version__}")

# Check available devices
print(f"Available devices: {jax.devices()}")

# A simple computation to test
x = jnp.array([1.0, 2.0, 3.0])
y = x * 2
print(f"Test computation: {x} * 2 = {y}")

print("\n--- Detailed JAX and Jaxlib information ---")
!pip show jax jaxlib

print("\n--- Detailed Optax information ---")
!pip show optax


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D # Import for 3D plotting

# Define the function
def f(x, y):
    return jnp.sin(jnp.sqrt(x**2 + y**2))

# Create a meshgrid for x and y values
x_range = jnp.linspace(-6, 6, 100)
y_range = jnp.linspace(-6, 6, 100)
X, Y = jnp.meshgrid(x_range, y_range)

# Compute the Z values
Z = f(X, Y)

# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
ax.plot_surface(X, Y, Z, cmap='viridis')

# Set labels and title
ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Z axis')
ax.set_title('3D Surface Plot of z = sin(sqrt(x^2 + y^2))')

plt.show()

In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Jaxlib version: {jax.lib.__version__}")

# Check available devices
print(f"Available devices: {jax.devices()}")

# A simple computation to test
x = jnp.array([1.0, 2.0, 3.0])
y = x * 2
print(f"Test computation: {x} * 2 = {y}")

太棒了！jax.grad 的範例已成功執行。讓我們來解釋一下這些結果：

純量函數的梯度： 對於函數 my_function(x) = x^2 + 2x + 1，它的導數是 2x + 2。 當 x=3.0 時，my_function(3.0) 的輸出是 3^2 + 2*3 + 1 = 9 + 6 + 1 = 16.0。 而 grad_my_function(3.0) 也就是在 x=3.0 時的導數，計算結果是 2*3 + 2 = 8.0。這與我們手動計算的結果完全一致！

向量函數的梯度： 對於函數 sum_of_squares(x)，其中 x 是一個向量 [x1, x2, x3]，函數定義為 x1^2 + x2^2 + x3^2。 對於這個函數，對每個 xi 的偏導數是 2*xi。 當 x_vec = [1.0, 2.0, 3.0] 時，sum_of_squares(x_vec) 的輸出是 1^2 + 2^2 + 3^2 = 1 + 4 + 9 = 14.0。 grad_sum_of_squares(x_vec) 則返回一個包含每個元素偏導數的向量：[2*1, 2*2, 2*3] = [2.0, 4.0, 6.0]。這表明 jax.grad 也能夠處理多變量函數的梯度。

這個範例清晰地展示了 jax.grad 如何自動且正確地計算函數的梯度，無論是純量還是向量輸入。這是 JAX 在深度學習和數值優化中非常強大的一個功能。

接下來，我們將探討 JAX 的另一個重要特性：Just-In-Time 編譯 (jax.jit)，它可以顯著加速您的 JAX 程式。

非常好！jax.jit 的矩陣乘法範例已經成功執行，結果顯示 jax.jit 確實能帶來顯著的性能提升。

結果分析：

未經 JIT 編譯的時間: 0.0889 秒
JIT 編譯 (首次運行，包含編譯時間): 0.0575 秒
JIT 編譯 (第二次運行，已編譯): 0.0453 秒
從結果可以看出，即使是第一次運行（包含了編譯器將 Python/NumPy 程式碼轉換為優化 XLA 程式碼的開銷），JIT 版本也比非 JIT 版本快。更重要的是，第二次運行 JIT 編譯過的函數時，由於編譯已完成，執行時間顯著減少，達到了最快的 0.0453 秒。這清楚地展示了 jax.jit 在重複執行相同計算時的巨大優勢。同時，我們也驗證了兩種方法得到的結果是近似相等的 (True)，確保了計算的正確性。

接下來，我們將探討 JAX 的第三個核心特性：自動向量化 (jax.vmap)，它允許您將操作批次化，而無需手動調整程式碼。

### 將 JAX 程式碼擴展至神經網路訓練

JAX 在深度學習領域非常強大，它提供了一系列工具來高效地構建和訓練神經網路。我們之前學習的 `jax.grad` (自動微分)、`jax.jit` (即時編譯)、`jax.vmap` (自動向量化) 和 `jax.pmap` (自動並行化) 都是其核心。

一個典型的神經網路訓練流程包括以下幾個關鍵部分：

1.  **模型定義 (Model Definition)**：定義網路的架構，包括層次、激活函數和可訓練的參數。
2.  **損失函數 (Loss Function)**：量化模型預測與真實標籤之間的差異。
3.  **優化器 (Optimizer)**：根據損失函數的梯度來更新模型參數，以最小化損失。
4.  **訓練循環 (Training Loop)**：反覆執行前向傳播、計算損失、反向傳播計算梯度、以及使用優化器更新參數的過程。


#### 1. 模型定義

在 JAX 中，神經網路模型通常被表示為一個 Python 函數，該函數接收參數 (weights) 和輸入數據，並輸出預測結果。由於 JAX 的函數式編程特性，模型的參數通常作為函數的第一個參數傳入。

您可以手動定義層次，或者使用像 [Flax](https://flax.readthedocs.io/) 或 [Haiku](https://dm-haiku.readthedocs.io/) 這樣的 JAX 友好型神經網路函式庫，它們提供了更高級的抽象來簡化模型構建。對於簡單的範例，我們可以從手動定義開始。


#### 2. 損失函數

損失函數 (如均方誤差 MSE、交叉熵 Cross-Entropy) 是衡量模型預測質量的重要指標。在 JAX 中，這只是一個接受模型參數、輸入數據和真實標籤，然後輸出一個純量損失值的函數。`jax.grad` 將用於計算這個損失函數關於模型參數的梯度。


#### 3. 優化器

優化器負責根據損失函數的梯度來更新模型的參數。JAX 生態系中常用的優化器庫是 [Optax](https://optax.readthedocs.io/)，它提供了各種標準的優化演算法 (如 SGD, Adam, RMSprop 等)。

Optax 優化器通常有兩個主要部分：`init` 函數用於初始化優化器狀態，`update` 函數用於根據梯度更新參數和優化器狀態。


#### 4. 訓練循環

訓練循環是將上述組件整合起來的過程。它通常涉及：

-   **初始化**：初始化模型參數和優化器狀態。
-   **批次數據**：將訓練數據分成小批次 (mini-batches)。
-   **前向傳播**：使用當前參數計算模型在批次數據上的預測。
-   **計算損失**：比較預測和真實標籤，計算損失。
-   **反向傳播**：使用 `jax.grad` 計算損失關於模型參數的梯度。
-   **參數更新**：使用優化器和梯度來更新模型參數。
-   **JIT 編譯**：通常會將一個訓練步驟 (或整個訓練循環) 使用 `jax.jit` 包裹起來，以獲得最佳性能。

接下來，我們將通過一個簡單的全連接網路來演示這個過程，用於一個二元分類任務。

### 載入並預處理 MNIST 資料集

我們將使用 `tensorflow_datasets` 來方便地載入 MNIST 資料。載入後，我們會對圖像進行展平（flatten）和歸一化（normalize），並將其轉換為 JAX 可用的 `jnp.array`。

In [ ]:
# 安裝 tensorflow 和 tensorflow_datasets
!pip install -qqq tensorflow
!pip install -qqq tensorflow_datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import jax
import jax.numpy as jnp

# 確保 TensorFlow 不會佔用 GPU 記憶體，因為 JAX 需要使用
tf.config.set_visible_devices([], 'GPU')

# 載入 MNIST 訓練和測試資料集
def load_mnist_dataset():
    ds_train, ds_test = tfds.load(name="mnist", split=["train", "test"], as_supervised=True)

    # 預處理函數：展平圖像並歸一化
    def preprocess_image(image, label):
        image = tf.cast(image, tf.float32) / 255.0  # 歸一化到 0-1 範圍
        image = tf.reshape(image, (-1,))  # 展平圖像為一維向量 (28*28 = 784)
        return image, label

    # 應用預處理
    ds_train = ds_train.map(preprocess_image)
    ds_test = ds_test.map(preprocess_image)

    # 將 tf.data.Dataset 轉換為 numpy 陣列
    # 注意：這會將整個資料集載入到記憶體，對於大資料集可能需要分批處理
    train_images_np = jnp.array([x for x, _ in tfds.as_numpy(ds_train)])
    train_labels_np = jnp.array([y for _, y in tfds.as_numpy(ds_train)])
    test_images_np = jnp.array([x for x, _ in tfds.as_numpy(ds_test)])
    test_labels_np = jnp.array([y for _, y in tfds.as_numpy(ds_test)])

    return train_images_np, train_labels_np, test_images_np, test_labels_np

print("載入 MNIST 資料集...")
train_images, train_labels, test_images, test_labels = load_mnist_dataset()

print(f"訓練圖像形狀: {train_images.shape}, 訓練標籤形狀: {train_labels.shape}")
print(f"測試圖像形狀: {test_images.shape}, 測試標籤形狀: {test_labels.shape}")

# 確認輸入和輸出維度
mnist_input_dim = train_images.shape[-1]
mnist_output_dim = 10 # MNIST 有 10 個數字類別 (0-9)

print(f"MNIST 輸入維度: {mnist_input_dim}")
print(f"MNIST 輸出維度 (類別數): {mnist_output_dim}")


### 調整模型參數並重新訓練

現在我們將使用之前定義的模型和訓練流程，但會更新輸入和輸出維度以適應 MNIST 資料集。由於 MNIST 是多類別分類，我們的交叉熵損失函數 (`cross_entropy_loss`) 已經支援多類別 one-hot 編碼，所以無需修改。

我們將使用相同的模型結構 (兩層全連接網路)，但會增加 `hidden_dim` 以處理更複雜的輸入。


In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import optax

# 重新定義模型、損失和優化器 (與之前相同，但確保輸入/輸出尺寸能被覆蓋)

def init_params(key, input_dim, hidden_dim, output_dim):
    key1, key2 = random.split(key)
    params = {
        'w1': random.normal(key1, (input_dim, hidden_dim)) * 0.01,
        'b1': jnp.zeros(hidden_dim),
        'w2': random.normal(key2, (hidden_dim, output_dim)) * 0.01,
        'b2': jnp.zeros(output_dim),
    }
    return params

def predict(params, x):
    h = jnp.dot(x, params['w1']) + params['b1']
    h = jax.nn.relu(h)
    logits = jnp.dot(h, params['w2']) + params['b2']
    return logits

def cross_entropy_loss(logits, labels):
    one_hot_labels = jax.nn.one_hot(labels, num_classes=logits.shape[-1])
    log_probs = jax.nn.log_softmax(logits)
    loss = -jnp.sum(one_hot_labels * log_probs, axis=-1)
    return jnp.mean(loss)

def loss_fn(params, x, y_true):
    y_pred_logits = predict(params, x)
    loss = cross_entropy_loss(y_pred_logits, y_true)
    return loss

learning_rate = 0.001 # 對於 MNIST，學習率可能需要調整
optimizer = optax.adam(learning_rate)

@jax.jit
def train_step(params, opt_state, x_batch, y_batch):
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

@jax.jit
def evaluate(params, x_test, y_test):
    logits = predict(params, x_test)
    predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(predictions == y_test)
    return accuracy

# --- 使用 MNIST 數據的訓練參數 ---
input_dim = mnist_input_dim # 784
hidden_dim = 256 # 增加隱藏層大小
output_dim = mnist_output_dim # 10

num_epochs = 10
batch_size = 128 # 增加批次大小

key = random.PRNGKey(0)
key, init_key = random.split(key)
params = init_params(init_key, input_dim, hidden_dim, output_dim)
opt_state = optimizer.init(params)

print("\n開始使用 MNIST 資料集訓練神經網路...")

# 訓練循環
num_train_samples = train_images.shape[0]

for epoch in range(num_epochs):
    key, shuffle_key = random.split(key)
    permutation = random.permutation(shuffle_key, num_train_samples)
    X_shuffled = train_images[permutation]
    Y_shuffled = train_labels[permutation]

    epoch_loss = 0.0
    num_batches = num_train_samples // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size
        x_batch = X_shuffled[start_idx:end_idx]
        y_batch = Y_shuffled[start_idx:end_idx]

        params, opt_state, loss = train_step(params, opt_state, x_batch, y_batch)
        epoch_loss += loss

    avg_epoch_loss = epoch_loss / num_batches
    train_accuracy = evaluate(params, train_images, train_labels)
    test_accuracy = evaluate(params, test_images, test_labels)

    print(f"Epoch {epoch + 1}/{num_epochs}, Avg Loss: {avg_epoch_loss:.4f}, Train Acc: {train_accuracy:.4f}, Test Acc: {test_accuracy:.4f}")

print("\nMNIST 神經網路訓練完成！")

final_test_accuracy = evaluate(params, test_images, test_labels)
print(f"最終測試集準確度: {final_test_accuracy * 100:.2f}%")


In [ ]:
import jax
import jax.numpy as jnp
from jax import random

# 假設我們將使用 Optax 作為優化器。如果尚未安裝，請先安裝：
# !pip install -qqq optax
import optax

print(f"JAX version: {jax.__version__}")
print(f"Optax version: {optax.__version__}")

# 1. 模型定義：簡單的兩層全連接網路
def init_params(key, input_dim, hidden_dim, output_dim):
    key1, key2 = random.split(key)
    params = {
        'w1': random.normal(key1, (input_dim, hidden_dim)) * 0.01,
        'b1': jnp.zeros(hidden_dim),
        'w2': random.normal(key2, (hidden_dim, output_dim)) * 0.01,
        'b2': jnp.zeros(output_dim),
    }
    return params

def predict(params, x):
    # First layer
    h = jnp.dot(x, params['w1']) + params['b1']
    h = jax.nn.relu(h) # ReLU activation
    # Output layer
    logits = jnp.dot(h, params['w2']) + params['b2']
    return logits

# 2. 損失函數：交叉熵損失 (用於二元分類)
def cross_entropy_loss(logits, labels):
    # 將 logits 轉換為機率，然後計算交叉熵
    one_hot_labels = jax.nn.one_hot(labels, num_classes=logits.shape[-1])
    # Optax 提供了 softmax_cross_entropy_with_integer_labels，但我們手動實現以示範
    # softmax_cross_entropy_with_integer_labels 已經包含了 log_softmax
    log_probs = jax.nn.log_softmax(logits)
    loss = -jnp.sum(one_hot_labels * log_probs, axis=-1)
    return jnp.mean(loss)

# 組合模型預測和損失計算的函數，方便 jax.grad 使用
def loss_fn(params, x, y_true):
    y_pred_logits = predict(params, x)
    loss = cross_entropy_loss(y_pred_logits, y_true)
    return loss

# 3. 優化器：使用 Adam 優化器
learning_rate = 0.01
optimizer = optax.adam(learning_rate)

# 4. 訓練步驟函數
@jax.jit
def train_step(params, opt_state, x_batch, y_batch):
    # 計算損失和梯度
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)

    # 使用優化器更新參數
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

# --- 準備數據 (簡單的合成數據用於二元分類) ---
# 假設輸入是 2 維，輸出是 2 個類別
input_dim = 2
hidden_dim = 10
output_dim = 2 # 二元分類 (0 或 1)
num_samples = 100

key = random.PRNGKey(0)
key, subkey = random.split(key)
X_data = random.normal(subkey, (num_samples, input_dim)) * 2

# 創建一些簡單的線性可分數據作為標籤
Y_data = (X_data[:, 0] + X_data[:, 1] > 0).astype(int)

# 訓練參數
num_epochs = 50
batch_size = 16

# 初始化模型參數和優化器狀態
key, init_key = random.split(key)
params = init_params(init_key, input_dim, hidden_dim, output_dim)
opt_state = optimizer.init(params)

print("\n開始訓練神經網路...")

# 訓練循環
for epoch in range(num_epochs):
    key, shuffle_key = random.split(key)
    # 隨機打亂數據
    permutation = random.permutation(shuffle_key, num_samples)
    X_shuffled = X_data[permutation]
    Y_shuffled = Y_data[permutation]

    epoch_loss = 0.0
    num_batches = num_samples // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size
        x_batch = X_shuffled[start_idx:end_idx]
        y_batch = Y_shuffled[start_idx:end_idx]

        params, opt_state, loss = train_step(params, opt_state, x_batch, y_batch)
        epoch_loss += loss

    avg_epoch_loss = epoch_loss / num_batches
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{num_epochs}, Average Loss: {avg_epoch_loss:.4f}")

print("\n訓練完成！")

# 評估模型
@jax.jit
def evaluate(params, x_test, y_test):
    logits = predict(params, x_test)
    predictions = jnp.argmax(logits, axis=-1)
    accuracy = jnp.mean(predictions == y_test)
    return accuracy

accuracy = evaluate(params, X_data, Y_data) # 使用全部數據進行簡單評估
print(f"訓練數據上的最終準確度: {accuracy * 100:.2f}%")


### JAX 與 TPU (Tensor Processing Unit)

TPU 是 Google 專為機器學習工作負載設計的 ASIC。JAX 能夠無縫利用 TPU 的強大平行運算能力，以加速訓練和推斷。要在 Colab 中使用 TPU，您需要將運行時類型更改為 TPU。

**請先確保您的 Colab 運行時已設定為 TPU (Runtime -> Change runtime type -> Hardware accelerator: TPU)。**

In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Jaxlib version: {jax.lib.__version__}")

# 檢查可用的設備
# 如果運行時設置為TPU，這裡應該會列出多個TPU設備（例如：8個TPU核心）
devices = jax.devices()
print(f"Available devices: {devices}")
print(f"Number of devices: {len(devices)}")

if any(device.platform == 'tpu' for device in devices):
    print("JAX has detected TPU devices!")
else:
    print("Warning: No TPU devices detected. Please ensure your Colab runtime is set to TPU.")

# 在設備上執行一個簡單的計算
# 為了在TPU上進行有效的計算，通常我們會使用JIT編譯
@jax.jit
def tpu_computation(x, y):
    return jnp.dot(x, y)

# 創建一些隨機數據
key = jax.random.PRNGKey(0)
size = 2000 # 稍微增大矩陣大小，以便在TPU上看到潛在的優勢
matrix_a = jax.random.normal(key, (size, size))
matrix_b = jax.random.normal(key, (size, size))

print(f"\nPerforming {size}x{size} matrix multiplication on detected device...")

# 執行計算，JAX會自動將計算發送到TPU（如果可用）
result_tpu = tpu_computation(matrix_a, matrix_b).block_until_ready()

print(f"Computation completed. Result shape: {result_tpu.shape}")
print(f"First few elements of result:\n{result_tpu[:2, :2]}")

print("\n這顯示 JAX 已成功在您的設備（希望是 TPU）上執行了計算。")

## JAX Tutorial Summary

這段對話涵蓋了在 Google Colab 中學習和使用 JAX 的多個方面，從基礎安裝到高級應用。

**對話摘要：**

1.  **JAX 核心特性教學 (Initials JAX Tutorial):**
    *   **需求:** 使用者要求在 Colab 中提供 JAX 教學，包含安裝步驟、展示 JAX 特性的範例和程式碼說明。
    *   **行動:** AI 提供了 JAX 的簡介，並逐步展示了其四大核心功能：
        *   **`jax.grad` (自動微分):** 示範了純量和向量函數的梯度計算，結果正確。
        *   **`jax.jit` (即時編譯):** 透過 1000x1000 矩陣乘法展示了顯著的性能加速，特別是在重複運行時。
        *   **`jax.vmap` (自動向量化):** 透過點積運算示範了如何自動處理批次數據而無需手動迴圈。
        *   **`jax.pmap` (自動並行化):** 介紹了其多設備並行能力。然而，由於 Colab 環境僅檢測到一個 CPU 設備，`pmap` 在此環境下表現得類似 `jit`，著重於單設備的編譯和優化，未能展示真正的數據並行優勢。
    *   **決策:** 使用者在每個範例完成後都批准了程式碼和解釋。

2.  **JAX 與 TPU 整合範例:**
    *   **需求:** 使用者要求展示 JAX 在 TPU 上的範例。
    *   **行動:** AI 指導使用者將 Colab 運行時更改為 TPU，並提供程式碼驗證 TPU 設備並執行 2000x2000 矩陣乘法。
    *   **結果:** TPU 成功被檢測到並執行了計算，但 JAX/Jaxlib 版本在 TPU 環境中顯示為 0.7.2（與之前 CPU 環境的 0.9.2 不同），且出現了關於「透明巨頁」的 `UserWarning`，但這不影響功能。

3.  **JAX 學習筆記總結:**
    *   **需求:** 使用者要求將對話紀錄整理成完整的 JAX 學習筆記。
    *   **行動:** AI 更新了筆記本中現有的總結部分，納入了 TPU 的內容，形成了一個涵蓋所有已討論 JAX 特性的綜合筆記。

4.  **擴展至神經網路訓練 (MNIST):**
    *   **需求:** 使用者要求解釋如何將 JAX 程式碼擴展至神經網路訓練，並提供 MNIST 範例。
    *   **行動:** AI 提供了神經網路訓練的關鍵概念（模型定義、損失函數、優化器、訓練循環），並嘗試提供一個使用 `tensorflow_datasets` 載入 MNIST 數據並進行訓練的範例。
    *   **問題 (未解決):**
        *   在載入 MNIST 數據時，最初因為 `tensorflow` 套件未明確安裝而導致 `ModuleNotFoundError` 和後續的 `NameError`。AI 嘗試透過在安裝 `tensorflow_datasets` 之前先安裝 `tensorflow` 來修復。
        *   後續在執行 MNIST 訓練程式碼和之前的 3D 繪圖程式碼時，反覆出現 `AttributeError: partially initialized module 'jax'`。AI 多次嘗試不同的 JAX/Jaxlib 安裝策略（如 `jax[cuda12_pip]`、明確版本號）來解決依賴衝突，但問題持續存在。

5.  **JAX 3D 功能與視覺化:**
    *   **需求:** 使用者詢問 JAX 的 3D 功能，並要求使用 Matplotlib 繪製 3D 曲面圖。
    *   **行動:** AI 解釋了 JAX 本身不是 3D 繪圖庫，但作為數值運算後端可以與其他 Python 3D 視覺化套件（如 Matplotlib, Plotly, Open3D）結合。
    *   **行動:** 提供了使用 Matplotlib 繪製 `z = x^2 - y^2` 和 `z = sin(sqrt(x^2 + y^2))` 3D 曲面圖的程式碼。
    *   **問題 (未解決):** 這些 3D 繪圖範例的執行也因上述持續的 `AttributeError: partially initialized module 'jax'` 而失敗。

**未解決的問題或任務：**

*   **JAX 環境問題:** 最關鍵且尚未解決的問題是**持續出現的 `AttributeError: partially initialized module 'jax'`**。這表明 JAX 和 `jaxlib` 的安裝或 Colab 環境本身處於不穩定狀態，阻礙了所有 JAX 程式碼的正常執行。AI 數次嘗試透過建議使用者**重啟 Colab 運行時 (Runtime -> Restart runtime)** 來徹底清除環境，這是目前解決此問題的唯一推薦方案。
*   **MNIST 訓練結果解釋 (Blocked):** 由於 JAX 環境問題，MNIST 訓練的最終結果和解釋目前被阻塞。
*   **3D 視覺化範例執行 (Blocked):** 同樣因 JAX 環境問題，3D 繪圖範例的執行也被阻塞。

**總結來說，對話成功引導使用者了解了 JAX 的核心概念和應用，但在進一步的高級應用（如 MNIST 訓練和 3D 視覺化）中遇到了持續的 JAX 安裝/環境問題，需要透過重啟 Colab 運行時來解決。**

### 4. `jax.pmap`: 自動並行化 (Automatic Parallelization)

`jax.pmap` (parallel map) 是 JAX 的一個高階轉換，旨在自動在多個設備（如多個 GPU 或 TPU 核心）上並行執行計算。它實現了一種叫做「SPMD」(Single Program, Multiple Data) 的並行模式，即在所有設備上運行相同的程式，但每個設備處理不同的數據切片。這對於擴展大型模型和數據集上的訓練非常有用。

`pmap` 的主要特點：
-   **多設備並行**：允許您的 JAX 代碼利用多個加速器，自動處理數據分佈和結果聚合。
-   **SPMD 編程模型**：您編寫一次程式碼，它會被複製到每個設備上，並對分配給該設備的數據片段執行操作。
-   **通訊高效**：JAX 的 XLA 後端能夠優化設備之間的通訊，減少並行處理的開銷。

**使用注意事項**：
-   需要有多個可用的設備（例如，Colab Pro 或 TPU 運行時提供多個 TPU 核心）。在單個 CPU 或 GPU 上運行 `pmap` 仍然可以工作，但可能不會顯示真正的性能優勢，因為它會模擬並行。
-   `in_axes` 和 `out_axes` 參數與 `vmap` 類似，用於指定數據在設備之間如何分佈以及結果如何組合。
-   `pmap` 會將輸入數據在指定的軸上切分到各個設備，並在輸出時在同一軸上合併結果。

我們將使用一個簡單的矩陣乘法來展示 `jax.pmap` 的基本用法。為了看到實際的性能提升，您需要在多設備環境下運行此範例（例如，Colab 的 TPU 運行時）。

In [ ]:
import jax
import jax.numpy as jnp
import time

# Check if multiple devices are available
num_devices = jax.device_count()
print(f"Available devices for pmap: {jax.devices()}")
print(f"Number of devices: {num_devices}")

if num_devices < 2:
    print("Warning: jax.pmap typically shows performance benefits with 2 or more devices (e.g., multiple GPUs or TPU cores).")
    print("Running pmap on a single device will still work but might not show speedup.")

# Define a function to be pmap'd
def pmap_matrix_multiply(x_slice, y_slice):
    return jnp.dot(x_slice, y_slice)

# Create large matrices
size = 1000
key = jax.random.PRNGKey(0)

# Create data that can be sharded across devices
# For pmap, we need to create an initial batch dimension for sharding
# Let's say we want to shard `matrix_a` along its first dimension.
# To keep it simple, we'll create identical slices for demonstration.
# In a real scenario, you'd shard a larger dataset.

# We'll simulate sharding by creating a batch dimension that matches num_devices
# Each device gets a slice of A and a full Y for a simple example.

matrix_a_batched = jnp.stack([jax.random.normal(key, (size, size)) for _ in range(num_devices)])
matrix_b = jax.random.normal(key, (size, size))

print(f"Performing {size}x{size} matrix multiplication using pmap...")

# Apply pmap to the function
# in_axes=(0, None) means:
#   - for the first argument (matrix_a_batched), shard along axis 0 (each device gets a different slice).
#   - for the second argument (matrix_b), replicate it to all devices (each device gets the full matrix_b).
# out_axes=0 means the outputs from each device will be stacked along axis 0.
pmap_multiply_func = jax.pmap(pmap_matrix_multiply, in_axes=(0, None))

start_time = time.time()
# Execute the pmap'd function
# The output will have a leading batch dimension corresponding to the devices
result_pmap = pmap_multiply_func(matrix_a_batched, matrix_b).block_until_ready()
end_time = time.time()
print(f"Time with JAX pmap: {end_time - start_time:.4f} seconds")

# Verify the shape of the result
print(f"Shape of pmap result: {result_pmap.shape}")

# For comparison, let's do a sequential calculation (if num_devices > 1)
if num_devices > 1:
    # Reshape matrix_a_batched to simulate the original full matrix A for comparison
    # This is a conceptual comparison, as pmap is for parallelizing a single large computation.
    # For a fair comparison, one would usually compare against a jit'd non-pmap version of the full op.

    # Here we simulate aggregating results from individual device computations
    # This comparison is indicative, but direct speedup comparison can be complex due to data transfer.
    print("\nPerforming sequential calculation for comparison...")
    start_time_seq = time.time()
    sequential_results = []
    for i in range(num_devices):
        slice_a = matrix_a_batched[i]
        seq_res = jnp.dot(slice_a, matrix_b).block_until_ready()
        sequential_results.append(seq_res)
    sequential_output = jnp.stack(sequential_results)
    end_time_seq = time.time()
    print(f"Time sequential (simulated) for {num_devices} operations: {end_time_seq - start_time_seq:.4f} seconds")
    print(f"Results are approximately equal (pmap vs. sequential): {jnp.allclose(result_pmap, sequential_output)}")


### 3. `jax.vmap`: 自動向量化 (Automatic Vectorization)

`jax.vmap` 是一個函數轉換 (function transformation)，它允許您自動地將一個操作單個輸入的函數轉換為一個操作批次輸入的函數。這在處理批次數據時非常有用，可以避免手動編寫迴圈，並提高效率。`vmap` 可以在不需要重新編寫邏輯的情況下，將元素級操作提升到批次級操作。

例如，假設我們有一個函數用於計算兩個向量的點積。如果我們想計算一個向量與多個向量的點積，我們可以使用 `vmap` 而無需顯式地遍歷這些向量。

In [ ]:
import jax
import jax.numpy as jnp

# Define a function that operates on single inputs
def dot_product(x, y):
    return jnp.dot(x, y)

# Example usage for a single pair of vectors
vec1 = jnp.array([1.0, 2.0, 3.0])
vec2 = jnp.array([4.0, 5.0, 6.0])
single_dot = dot_product(vec1, vec2)
print(f"Single dot product: {single_dot}")

# Now, let's say we have multiple vectors (batch_of_vecs) and we want to dot_product each with a single vector (vec2).
# We can use jax.vmap to batch this operation.

batch_of_vecs = jnp.array([
    [1.0, 2.0, 3.0],
    [7.0, 8.0, 9.0],
    [0.0, 1.0, 0.0]
])

# vmap will apply dot_product to each row of batch_of_vecs with vec2.
# in_axes=(0, None) means:
#   - for the first argument (batch_of_vecs), batch along axis 0 (rows).
#   - for the second argument (vec2), don't batch (keep it as is for every operation).
# out_axes=0 means the output will have a new batch dimension at axis 0.
batched_dot_product = jax.vmap(dot_product, in_axes=(0, None))

results = batched_dot_product(batch_of_vecs, vec2)
print(f"\nBatched dot products:\n{results}")

# Verify manually:
# (1*4 + 2*5 + 3*6) = 4 + 10 + 18 = 32
# (7*4 + 8*5 + 9*6) = 28 + 40 + 54 = 122
# (0*4 + 1*5 + 0*6) = 0 + 5 + 0 = 5
# The results should be [32., 122., 5.]

# Another example: batching both inputs
# If we have two batches of vectors and want to compute element-wise dot products between corresponding vectors
batch_of_vecs_a = jnp.array([
    [1.0, 2.0],
    [3.0, 4.0]
])
batch_of_vecs_b = jnp.array([
    [5.0, 6.0],
    [7.0, 8.0]
])

# in_axes=(0, 0) means batch along axis 0 for both arguments.
batched_dot_product_both = jax.vmap(dot_product, in_axes=(0, 0))
results_both = batched_dot_product_both(batch_of_vecs_a, batch_of_vecs_b)
print(f"\nBatched dot products (both inputs batched):\n{results_both}")

# Verify manually:
# (1*5 + 2*6) = 5 + 12 = 17
# (3*7 + 4*8) = 21 + 32 = 53
# The results should be [17., 53.]